In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def load_and_aggregate_data(file_path):
    """Load data and aggregate across the 3 runs for each configuration."""
    # Load CSV file (comma-separated)
    df = pd.read_csv(file_path)
    
    # Print info for debugging
    print(f"\nColumns found: {list(df.columns)}")
    print(f"Shape: {df.shape}")
    
    # Filter to only batch size 64
    print("\nFiltering to batch size 64 only...")
    df = df[df['batch_size'] == 64]
    print(f"After filtering: {df.shape}")
    
    # Check if data is already aggregated or needs aggregation
    if 'n_runs' in df.columns and df['n_runs'].iloc[0] == 3:
        print("\nData appears to already be aggregated (n_runs = 3)")
        print("Using data as-is without further aggregation")
        aggregated = df.copy()
    else:
        print("\nAggregating data across runs...")
        # Group by model configuration and aggregate
        group_cols = ['model_name', 'model_type', 'batch_size', 'precision']
        
        agg_dict = {
            'mean_accuracy': 'mean',
            'mean_auc': 'mean',
            'mean_throughput': 'mean',
            'mean_energy_per_img': 'mean',
            'mean_latency_ms': 'mean',
            'mean_peak_memory_mb': 'mean',
            'model_size_mb': 'first',
            'total_params': 'first'
        }
        
        aggregated = df.groupby(group_cols).agg(agg_dict).reset_index()
    
    # Create display name and grouping info
    aggregated['base_model'] = aggregated['model_name'].apply(extract_base_model)
    aggregated['display_name'] = aggregated.apply(create_display_name, axis=1)
    
    return aggregated

def extract_base_model(model_name):
    """Extract the base model size (base, small, tiny) from model name."""
    if 'base' in model_name:
        return 'base'
    elif 'small' in model_name:
        return 'small'
    elif 'tiny' in model_name:
        return 'tiny'
    return 'unknown'

def create_display_name(df_row):
    """Create a readable display name for each model."""
    name = df_row['model_name']
    model_type = df_row['model_type']
    
    # Simplify name
    name = name.replace('vit_', '').replace('_patch16_224', '')
    
    if model_type == 'baseline':
        return f"{name}"
    elif model_type == 'quantized':
        return f"{name} (quant)"
    elif model_type == 'kd':
        # Extract teacher info
        if 'from' in name:
            parts = name.split('_kd_from_')
            student = parts[0]
            teacher = parts[1].replace('vit_', '')
            return f"{student} (KD from {teacher})"
        return f"{name} (KD)"
    
    return name

def get_models_for_group(df, group_key):
    """Get the relevant models for each comparison group."""
    
    if group_key == 'group1_baseline_family':
        # Just the three baseline models
        return df[
            (df['model_name'].isin(['vit_base_patch16_224', 'vit_small_patch16_224', 'vit_tiny_patch16_224'])) &
            (df['model_type'] == 'baseline')
        ].copy()
    
    elif group_key == 'group2a_base_kd_chain':
        # Base (teacher) + Small KD from Base + Tiny KD from Base
        models = [
            'vit_base_patch16_224',
            'vit_small_patch16_224_kd_from_vit_base_patch16_224',
            'vit_tiny_patch16_224_kd_from_vit_base_patch16_224'
        ]
        return df[df['model_name'].isin(models)].copy()
    
    elif group_key == 'group2b_small_kd_chain':
        # Small (teacher) + Tiny KD from Small
        models = [
            'vit_small_patch16_224',
            'vit_tiny_patch16_224_kd_from_vit_small_patch16_224'
        ]
        return df[
            (df['model_name'].isin(models)) &
            ((df['model_type'] == 'baseline') | (df['model_type'] == 'kd'))
        ].copy()
    
    elif group_key == 'group3_baseline_vs_quantized':
        # All baseline models + their quantized versions
        baseline_models = ['vit_tiny_patch16_224', 'vit_small_patch16_224', 'vit_base_patch16_224']
        quantized_models = ['vit_tiny_patch16_224_amp', 'vit_small_patch16_224_amp', 'vit_base_patch16_224_amp']
        
        return df[
            ((df['model_name'].isin(baseline_models)) & (df['model_type'] == 'baseline')) |
            ((df['model_name'].isin(quantized_models)) & (df['model_type'] == 'quantized'))
        ].copy()
    
    return pd.DataFrame()

def plot_comparison_groups(df, output_dir):
    """Create separate plots for each comparison group."""
    groups = {
        'group1_baseline_family': 'Pure Baseline Models (Base, Small, Tiny)',
        'group2a_base_kd_chain': 'Base Model Knowledge Distillation Chain',
        'group2b_small_kd_chain': 'Small Model Knowledge Distillation Chain',
        'group3_baseline_vs_quantized': 'Baseline vs Quantized Comparison'
    }
    
    for group_key, group_title in groups.items():
        group_data = get_models_for_group(df, group_key)
        
        if len(group_data) == 0:
            print(f"  Skipping {group_title} - no data")
            continue
        
        print(f"  Plotting {group_title} ({len(group_data)} configs)")
        print(f"    Models: {group_data['display_name'].tolist()}")
        
        # Create a comprehensive comparison figure
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(f"{group_title} (Batch Size 64)", fontsize=16, fontweight='bold', y=0.995)
        
        # Create consistent colors for each model
        unique_names = group_data['display_name'].unique()
        color_map = {name: plt.cm.Set3(i/len(unique_names)) for i, name in enumerate(unique_names)}
        colors = [color_map[name] for name in group_data['display_name']]
        
        # Plot 1: Throughput
        ax = axes[0, 0]
        x_pos = range(len(group_data))
        bars = ax.bar(x_pos, group_data['mean_throughput'], color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_data['display_name'], rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Throughput (img/s)', fontweight='bold', fontsize=11)
        ax.set_title('Throughput Comparison', fontweight='bold', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # Plot 2: Energy Efficiency
        ax = axes[0, 1]
        energy_nj = group_data['mean_energy_per_img'] * 1e9
        bars = ax.bar(x_pos, energy_nj, color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_data['display_name'], rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Energy (nJ/img)', fontweight='bold', fontsize=11)
        ax.set_title('Energy Use (kWh/img)', fontweight='bold', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # Plot 3: Accuracy
        ax = axes[0, 2]
        bars = ax.bar(x_pos, group_data['mean_accuracy'] * 100, color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_data['display_name'], rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=11)
        ax.set_title('Accuracy', fontweight='bold', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # Plot 4: Model Size
        ax = axes[1, 0]
        bars = ax.bar(x_pos, group_data['model_size_mb'], color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_data['display_name'], rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Model Size (MB)', fontweight='bold', fontsize=11)
        ax.set_title('Model Size', fontweight='bold', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # Plot 5: AUC
        ax = axes[1, 1]
        bars = ax.bar(x_pos, group_data['mean_auc'] * 100, color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_data['display_name'], rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('AUC (%)', fontweight='bold', fontsize=11)
        ax.set_title('AUC Performance', fontweight='bold', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # Plot 6: Memory Usage
        ax = axes[1, 2]
        bars = ax.bar(x_pos, group_data['mean_peak_memory_mb'], color=colors, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(group_data['display_name'], rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Peak Memory (MB)', fontweight='bold', fontsize=11)
        ax.set_title('Memory Usage', fontweight='bold', fontsize=12)
        ax.grid(axis='y', alpha=0.3)
        
        # Add a shared legend at the bottom
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=color_map[name], edgecolor='black', linewidth=1.5, label=name) 
                          for name in unique_names]
        fig.legend(handles=legend_elements, loc='lower center', ncol=min(len(unique_names), 4), 
                  bbox_to_anchor=(0.5, -0.02), fontsize=10, frameon=True, shadow=True)
        
        plt.tight_layout(rect=[0, 0.02, 1, 0.98])
        filename = group_key.replace('group', 'comparison').replace('_', '_') + '.png'
        plt.savefig(output_dir / filename, dpi=300, bbox_inches='tight')
        plt.close()

def plot_precision_comparison(df, output_dir):
    """Compare AMP vs FP32 for each model."""
    print("  Plotting precision comparison (AMP vs FP32)")
    
    # Get unique model configurations (ignoring precision)
    unique_models = df.groupby(['model_name', 'model_type']).size().reset_index()[['model_name', 'model_type']]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Precision Comparison: AMP vs FP32 (Batch Size 64)', fontsize=16, fontweight='bold')
    
    metrics = [
        ('mean_throughput', 'Throughput (img/s)', axes[0, 0]),
        ('mean_energy_per_img', 'Energy per Image (nJ)', axes[0, 1]),
        ('mean_latency_ms', 'Latency (ms)', axes[1, 0]),
        ('mean_peak_memory_mb', 'Peak Memory (MB)', axes[1, 1])
    ]
    
    for metric, ylabel, ax in metrics:
        amp_vals = []
        fp32_vals = []
        labels = []
        
        for _, row in unique_models.iterrows():
            amp_data = df[(df['model_name'] == row['model_name']) & 
                         (df['model_type'] == row['model_type']) & 
                         (df['precision'] == 'amp')]
            
            fp32_data = df[(df['model_name'] == row['model_name']) & 
                          (df['model_type'] == row['model_type']) & 
                          (df['precision'] == 'fp32')]
            
            if len(amp_data) > 0 and len(fp32_data) > 0:
                val_amp = amp_data[metric].values[0]
                val_fp32 = fp32_data[metric].values[0]
                
                if metric == 'mean_energy_per_img':
                    val_amp *= 1e9
                    val_fp32 *= 1e9
                
                amp_vals.append(val_amp)
                fp32_vals.append(val_fp32)
                
                short_name = row['model_name'].replace('vit_', '').replace('_patch16_224', '')[:20]
                labels.append(short_name)
        
        x = np.arange(len(labels))
        width = 0.35
        
        ax.bar(x - width/2, amp_vals, width, label='AMP', alpha=0.8, color='#3b82f6', edgecolor='black', linewidth=1.5)
        ax.bar(x + width/2, fp32_vals, width, label='FP32', alpha=0.8, color='#f59e0b', edgecolor='black', linewidth=1.5)
        
        ax.set_ylabel(ylabel, fontweight='bold', fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
        ax.legend(loc='best', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'precision_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

def plot_batch_size_impact(df, output_dir):
    """This function is now disabled since we only plot batch size 64."""
    print("  Skipping batch size impact (only batch size 64 in data)")
    return

def plot_efficiency_frontier(df, output_dir):
    """Plot Pareto frontier showing trade-offs."""
    print("  Plotting efficiency frontiers")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Efficiency Frontiers (Batch Size 64, AMP Precision)', fontsize=16, fontweight='bold')
    
    # Filter to AMP precision only for clarity
    df_amp = df[df['precision'] == 'amp'].copy()
    df_amp['energy_nj'] = df_amp['mean_energy_per_img'] * 1e9
    
    # Create group assignment for visualization
    def assign_group_for_viz(row):
        model_name = row['model_name']
        model_type = row['model_type']
        
        if model_type == 'baseline' and model_name in ['vit_base_patch16_224', 'vit_small_patch16_224', 'vit_tiny_patch16_224']:
            return 'Baseline Family'
        elif model_type == 'kd' and 'base' in model_name:
            return 'Base KD Chain'
        elif model_type == 'kd' and 'small' in model_name:
            return 'Small KD Chain'
        elif model_type == 'quantized':
            return 'Quantized'
        return 'Other'
    
    df_amp['viz_group'] = df_amp.apply(assign_group_for_viz, axis=1)
    
    group_colors = {
        'Baseline Family': '#3b82f6',
        'Base KD Chain': '#10b981',
        'Small KD Chain': '#8b5cf6',
        'Quantized': '#f59e0b',
        'Other': '#6366f1'
    }
    
    # Plot 1: Accuracy vs Energy
    ax = axes[0]
    for group in df_amp['viz_group'].unique():
        if group == 'Other':
            continue
        subset = df_amp[df_amp['viz_group'] == group]
        ax.scatter(subset['energy_nj'], subset['mean_accuracy'] * 100, 
                  s=150, alpha=0.7, label=group,
                  color=group_colors.get(group, '#6366f1'),
                  edgecolors='black', linewidth=1.5)
    
    ax.set_xlabel('Energy per Image (nJ)', fontweight='bold', fontsize=12)
    ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
    ax.set_title('Accuracy vs Energy Trade-off', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10, frameon=True, shadow=True)
    ax.grid(alpha=0.3)
    
    # Plot 2: Throughput vs Model Size
    ax = axes[1]
    for group in df_amp['viz_group'].unique():
        if group == 'Other':
            continue
        subset = df_amp[df_amp['viz_group'] == group]
        ax.scatter(subset['model_size_mb'], subset['mean_throughput'], 
                  s=150, alpha=0.7, label=group,
                  color=group_colors.get(group, '#6366f1'),
                  edgecolors='black', linewidth=1.5)
    
    ax.set_xlabel('Model Size (MB)', fontweight='bold', fontsize=12)
    ax.set_ylabel('Throughput (img/s)', fontweight='bold', fontsize=12)
    ax.set_title('Throughput vs Model Size Trade-off', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10, frameon=True, shadow=True)
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_dir / 'efficiency_frontiers.png', dpi=300, bbox_inches='tight')
    plt.close()

def print_summary_statistics(df):
    """Print summary statistics."""
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    
    print(f"\nTotal model configurations: {len(df)}")
    
    # Show breakdown by model type
    print(f"\nBy model type:")
    for model_type in df['model_type'].unique():
        count = len(df[df['model_type'] == model_type])
        print(f"  {model_type}: {count} configurations")
    
    print(f"\nBy precision:")
    for precision in df['precision'].unique():
        count = len(df[df['precision'] == precision])
        print(f"  {precision}: {count} configurations")
    
    print("\n" + "="*80 + "\n")

def main(data_path, output_dir='./visualizations'):
    """Main function to generate all visualizations."""
    # Create output directory
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    print(f"Loading data from: {data_path}")
    df = load_and_aggregate_data(data_path)
    
    print(f"\nProcessing {len(df)} model configurations")
    
    # Print summary statistics
    print_summary_statistics(df)
    
    # Generate all plots
    print("\nGenerating visualizations...")
    
    print("Creating comparison group plots...")
    plot_comparison_groups(df, output_dir)
    
    print("Creating precision comparison...")
    plot_precision_comparison(df, output_dir)
    
    print("Creating batch size impact analysis...")
    plot_batch_size_impact(df, output_dir)
    
    print("Creating efficiency frontiers...")
    plot_efficiency_frontier(df, output_dir)
    
    print(f"\nAll visualizations saved to: {output_dir.absolute()}")
    
    # Save aggregated data to CSV
    output_csv = output_dir / 'aggregated_results.csv'
    df.to_csv(output_csv, index=False)
    print(f"Aggregated data saved to: {output_csv}")


if __name__ == "__main__":
    # Set your input and output paths
    DATA_PATH = "/Users/arihangupta/Downloads/pruning_project_data/Vision/Benchmakring_results/dermamnist_summary.csv"
    OUTPUT_DIR = "/Users/arihangupta/Downloads/pruning_project_data/Vision/Benchmakring_results/visulizations/derma"
    
    main(DATA_PATH, output_dir=OUTPUT_DIR)

Loading data from: /Users/arihangupta/Downloads/pruning_project_data/Vision/Benchmakring_results/dermamnist_summary.csv

Columns found: ['dataset', 'model_name', 'model_type', 'batch_size', 'precision', 'n_runs', 'mean_accuracy', 'std_accuracy', 'mean_auc', 'std_auc', 'mean_throughput', 'std_throughput', 'median_throughput', 'mean_latency_ms', 'std_latency_ms', 'median_latency_ms', 'mean_peak_memory_mb', 'std_peak_memory_mb', 'mean_energy_kwh', 'mean_energy_per_img', 'mean_emissions_kg', 'model_size_mb', 'total_params']
Shape: (27, 23)

Filtering to batch size 64 only...
After filtering: (9, 23)

Data appears to already be aggregated (n_runs = 3)
Using data as-is without further aggregation

Processing 9 model configurations

SUMMARY STATISTICS

Total model configurations: 9

By model type:
  baseline: 3 configurations
  quantized: 3 configurations
  kd: 3 configurations

By precision:
  fp32: 6 configurations
  amp: 3 configurations



Generating visualizations...
Creating comparison 